# Project 4: Bank Client Risk Classifier
**Задача:** определить рискованных клиентов банка. Главная сложность — **несбалансированные классы** (рискованных мало).
**Темы:** imbalanced data, precision/recall, class_weight, сравнение моделей. Данные синтетические, проект учебный.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

rng = np.random.default_rng(33)
n = 2000
df = pd.DataFrame({
    "доход":            rng.normal(500, 180, n).clip(100),         # тыс. тг/мес
    "долг_к_доходу":    rng.uniform(0, 0.9, n).round(2),
    "просрочки_за_год": rng.poisson(0.6, n),
    "стаж_клиента_лет": rng.uniform(0, 15, n).round(1),
})
risk_score = (2.5*df["долг_к_доходу"] + 0.9*df["просрочки_за_год"]
              - 0.002*df["доход"] - 0.08*df["стаж_клиента_лет"] - 0.6)
df["риск"] = (rng.uniform(0, 1, n) < 1/(1+np.exp(-risk_score))).astype(int)
print("Баланс классов:"); print(df["риск"].value_counts(normalize=True).round(3))
df.head()

In [ ]:
X = df.drop(columns="риск"); y = df["риск"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Baseline-проверка: модель "все хорошие"
dummy = np.zeros(len(y_te))
print("Глупый baseline: accuracy =", (dummy == y_te).mean().round(3), "| но risk-клиентов найдено: 0")

# Модель 1: обычная логистическая регрессия
lr = Pipeline([("s", StandardScaler()), ("m", LogisticRegression())]).fit(X_tr, y_tr)
# Модель 2: с балансировкой классов — штраф за ошибки на редком классе выше
lr_bal = Pipeline([("s", StandardScaler()),
                   ("m", LogisticRegression(class_weight="balanced"))]).fit(X_tr, y_tr)
# Модель 3: random forest с балансировкой
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0).fit(X_tr, y_tr)

for name, m in [("LogReg", lr), ("LogReg balanced", lr_bal), ("RandomForest", rf)]:
    p = m.predict(X_te)
    print(f"\n=== {name} ===")
    print(classification_report(y_te, p, target_names=["надёжный", "риск"], digits=2))

In [ ]:
# Сравнение по матрицам ошибок: что важнее банку — не пропустить риск (recall класса 1)!
best = lr_bal
ConfusionMatrixDisplay(confusion_matrix(y_te, best.predict(X_te)),
                       display_labels=["надёжный", "риск"]).plot(cmap='Oranges')
plt.title('LogReg balanced'); plt.show()

# Важность признаков (по RF)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
plt.barh(imp.index, imp.values); plt.title('Что влияет на риск'); plt.grid(True, axis='x'); plt.show()

## Выводы
- Accuracy «глупой» модели высокая, а пользы ноль — на несбалансированных данных смотрим на recall/F1 редкого класса.
- class_weight="balanced" поднимает recall риска ценой precision — банку обычно выгоднее
  лишний раз проверить хорошего клиента, чем выдать кредит плохому (FN дороже FP).
- Долг/доход и просрочки — главные сигналы риска (ожидаемо и интерпретируемо).

## Задания
1. Через predict_proba и порог 0.3 подними recall риска ещё выше. Какой ценой?
2. Посчитай «деньги»: пусть FN стоит 1000$, FP стоит 50$. Какая модель дешевле банку?
3. Добавь roc_auc_score для всех трёх моделей.